# Result Availability Audit — Part 3 of 3
## Alert timing, lead time, and detection under the two clocks

Part 1 measured the interval: point of care results post at a median of 3 minutes, core laboratory
results at 60 minutes, and 9.0% of drawn results are still invisible to a model scoring at hour 6.
Part 2 turned that into a scoring problem and found that discrimination barely moves. The model
reaches AUROC 0.824 on the observation clock and 0.822 on the availability clock.

That near-null is the reason this notebook exists.

**Discrimination is the wrong measure for this question.** AUROC ranks every scoring hour against
every other. The availability clock does not mostly change *who* looks unwell, it changes *when* the
evidence arrives. A model can rank almost identically while its alert on a deteriorating patient
fires two hours later. Ranking metrics are blind to that; the patient is not.

Part 3 therefore measures what a deployed alerting system actually delivers:

| Outcome | Question it answers |
|---|---|
| Lead time before escalation | How much warning does the alert give? |
| Lead time lost, paired per patient | How much warning does the delay cost? |
| Detection status | Who is detected under one clock and not the other? |
| Alerts per 100 bed-days | What does the alerting cost the unit? |
| Number needed to alert | How many alerts buy one true detection? |
| Net benefit | Is the alert worth acting on across plausible thresholds? |
| Culture knowledge gap | How often does a result arrive after the event it should have predicted? |

**The primary comparison is a fixed threshold.** A threshold is chosen on the observation clock to
meet an alert budget, exactly as it would be during development, and then applied unchanged to the
availability clock. That is what deployment does. A workload-rematched threshold is reported as a
secondary analysis so that timing effects can be separated from volume effects.

Bedside scores act as the latency-insensitive reference throughout. Vital signs reach the record at
the moment of measurement, so a rule-based score loses no lead time by construction. Any lead time
the learned model loses is therefore attributable to the laboratory block alone.

**Runtime.** Five to fifteen minutes. Everything reads cached predictions; nothing touches a CSV.

---
## 1. Setup

In [1]:
import sys, os, json, time, hashlib, platform, warnings, gc
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, asdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

ENV_VERSIONS = {
    "python": platform.python_version(),
    "platform": f"{platform.system()} {platform.release()}",
    "pandas": pd.__version__, "numpy": np.__version__, "pyarrow": pa.__version__,
}
for k, v in ENV_VERSIONS.items():
    print(f"{k:<10} {v}")

python     3.11.5
platform   Windows 10
pandas     2.3.3
numpy      1.24.4
pyarrow    24.0.0


In [2]:
@dataclass(frozen=True)
class Config3:
    out_root: str = r"C:\Research_Paper_2\result_availability_audit"

    # ---- alerting ----------------------------------------------------------
    silence_hours: float = 4.0            # suppression window after an alert fires
    alert_budgets: tuple = (5, 10, 20, 40, 80)   # alerts per 100 occupied bed-days
    primary_budget: int = 20
    max_lead_hours: float = 48.0          # an alert further ahead than this is not credited
    min_lead_hours: float = 0.0

    # ---- net benefit -------------------------------------------------------
    nb_thresholds: tuple = tuple(np.round(np.arange(0.005, 0.101, 0.005), 4))

    # ---- microbiology ------------------------------------------------------
    culture_lookback_hours: float = 24.0  # a culture is "recent" within this window

    # ---- inference ---------------------------------------------------------
    n_boot: int = 1000                    # stay-level bootstrap replicates
    boot_seed: int = 20260906

    # ---- sensitivity -------------------------------------------------------
    silence_grid: tuple = (2.0, 4.0, 8.0)

    seed: int = 20260906

CFG = Config3()
OUT = Path(CFG.out_root)
DIR = {"cache": OUT / "cache", "tables": OUT / "tables", "figs": OUT / "figures"}
for d in DIR.values():
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(CFG.seed)
CFG_JSON = json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                       for k, v in asdict(CFG).items()}, sort_keys=True, default=str)
CFG_HASH = hashlib.sha256(CFG_JSON.encode()).hexdigest()

print("Output root :", OUT)
print("Config hash :", CFG_HASH[:16])
print(f"Alert budgets (per 100 bed-days): {CFG.alert_budgets}, primary = {CFG.primary_budget}")
print(f"Suppression window: {CFG.silence_hours:.0f} h")

Output root : C:\Research_Paper_2\result_availability_audit
Config hash : e8b65a35112b332a
Alert budgets (per 100 bed-days): (5, 10, 20, 40, 80), primary = 20
Suppression window: 4 h


In [3]:
# ---- provenance, chained onto Parts 1 and 2 -------------------------------
class Provenance:
    def __init__(self, seed_material):
        self.chain = hashlib.sha256(seed_material.encode()).hexdigest()
        self.records, self.timings = [], {}

    def add(self, role, name, path, extra=None):
        path = Path(path)
        if path.exists():
            h = hashlib.sha256()
            with open(path, "rb") as f:
                for b in iter(lambda: f.read(8 << 20), b""):
                    h.update(b)
            digest, size = h.hexdigest(), path.stat().st_size
        else:
            digest, size = "MISSING", 0
        rec = {"role": role, "name": name, "path": str(path), "bytes": size, "digest": digest,
               "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        if extra: rec.update(extra)
        self.chain = hashlib.sha256((self.chain + digest).encode()).hexdigest()
        rec["chain_after"] = self.chain
        self.records.append(rec)
        return rec

    def time(self, stage, seconds):
        self.timings[stage] = round(seconds, 2)

    def manifest(self):
        return {"generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "config": json.loads(CFG_JSON), "config_sha256": CFG_HASH,
                "environment": {**ENV_VERSIONS, "cpu_count": os.cpu_count()},
                "records": self.records, "timings": self.timings,
                "terminal_chain_sha256": self.chain}


PRIOR = ""
for nm in ("part1_manifest.json", "part2_manifest.json"):
    p = OUT / nm
    if p.exists():
        d = json.loads(p.read_text())
        PRIOR += d.get("terminal_chain_sha256", "")
        print(f"{nm}: {d.get('terminal_chain_sha256', '')[:32]}...")
    else:
        print(f"WARNING: {nm} not found. Run the earlier notebooks first.")

PROV = Provenance(CFG_HASH + PRIOR)


class Stage:
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.t0 = time.time(); print(f"[{self.label}] start"); return self
    def __exit__(self, *e):
        dt = time.time() - self.t0
        PROV.time(self.label, dt)
        print(f"[{self.label}] done in {dt:.1f} s")
        return False


def write_table(df, name, index=False):
    p = DIR["tables"] / f"{name}.csv"; df.to_csv(p, index=index)
    PROV.add("table", name, p, {"rows": int(len(df))}); return p

def save_fig(fig, name):
    p = DIR["figs"] / f"{name}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight", facecolor="white"); plt.close(fig)
    PROV.add("figure", name, p); return p

print("\nProvenance chain seeded:", PROV.chain[:16])

part1_manifest.json: c33a334d6c4d9b9a3c109f96753b8ccf...
part2_manifest.json: 47ff1b281a4b0ea4e60da8adcf9600a7...

Provenance chain seeded: 16f9279f501aff14


In [4]:
# ---- load cached predictions and supporting artefacts ---------------------
with Stage("load"):
    preds_m = pd.read_parquet(DIR["cache"] / "preds_mimic_test.parquet")
    preds_e = pd.read_parquet(DIR["cache"] / "preds_eicu_external.parquet")
    events_m = pd.read_parquet(DIR["cache"] / "events_mimic.parquet")
    events_e = pd.read_parquet(DIR["cache"] / "events_eicu.parquet")
    cohort_m = pd.read_parquet(DIR["cache"] / "cohort_mimic.parquet")
    cohort_e = pd.read_parquet(DIR["cache"] / "cohort_eicu.parquet")
    micro = pd.read_parquet(DIR["cache"] / "micro_mimic.parquet")

COHORTS = {
    "MIMIC-IV test":     {"preds": preds_m, "id": "stay_id",           "events": events_m},
    "eICU-CRD external": {"preds": preds_e, "id": "patientunitstayid", "events": events_e},
}

SCORE_COLS = {
    "model_obs":   "p_trained_obs__scored_obs",
    "model_avail": "p_trained_obs__scored_avail",
    "news2":       "news2",
    "qsofa":       "qsofa",
}

for tag, C in COHORTS.items():
    P, idc = C["preds"], C["id"]
    n_stay = P[idc].nunique()
    ev = P.loc[P["first_event_h"].notna(), idc].nunique()
    print(f"{tag:<20} {len(P):>10,} scoring hours | {n_stay:>7,} stays | "
          f"{ev:>6,} with an escalation ({100*ev/n_stay:.1f}%)")
    missing = [c for c in SCORE_COLS.values() if c not in P.columns]
    if missing:
        print(f"  MISSING COLUMNS: {missing}")

[load] start
[load] done in 0.6 s
MIMIC-IV test           180,139 scoring hours |   6,098 stays |    847 with an escalation (13.9%)
eICU-CRD external     1,892,310 scoring hours |  67,167 stays |  5,846 with an escalation (8.7%)


---
## 2. Alert simulation

A patient alerts when the score crosses the threshold. Once an alert fires it is suppressed for a
fixed window, because a system that re-fires every hour on the same patient is not delivering new
information and would inflate the burden count without adding clinical value.

Two quantities come out of this. The **first alert time** determines lead time and detection, and is
unaffected by suppression. The **alert count** determines workload, and depends on it entirely.

In [5]:
def alert_times(preds, idcol, score_col, thr, silence_h):
    """
    Returns (first_alert per stay, total alerts after suppression).
    Only the crossing rows are walked, so this stays fast at any cohort size.
    """
    hit = preds.loc[preds[score_col].to_numpy() >= thr, [idcol, "hour"]]
    if hit.empty:
        return pd.Series(dtype="float64", name="first_alert_h"), 0

    hit = hit.sort_values([idcol, "hour"])
    ids = hit[idcol].to_numpy()
    hrs = hit["hour"].to_numpy().astype("float64")

    keep = np.zeros(len(hrs), dtype=bool)
    last_id, last_h = None, -np.inf
    for i in range(len(hrs)):
        if ids[i] != last_id:
            last_id, last_h = ids[i], -np.inf
        if hrs[i] >= last_h + silence_h:
            keep[i] = True
            last_h = hrs[i]

    first = (pd.DataFrame({idcol: ids, "hour": hrs})
             .groupby(idcol)["hour"].min().rename("first_alert_h"))
    return first, int(keep.sum())


def bed_days(preds):
    """Occupied bed-days represented by the scoring grid."""
    return len(preds) / 24.0


def calibrate_threshold(preds, idcol, score_col, target_per_100_beddays, silence_h,
                        lo=None, hi=None, tol=0.02, max_iter=40):
    """Bisect on the score threshold until the suppressed alert rate meets the budget."""
    s = preds[score_col].dropna()
    if s.empty:
        return np.nan, np.nan
    bd = bed_days(preds)
    target = target_per_100_beddays * bd / 100.0

    # Bracket from the score distribution first. Suppressed alerts can never exceed raw
    # crossings, so a threshold admitting far more crossings than the budget is already
    # too low. Starting there keeps the suppression scan off the whole frame.
    if lo is None:
        frac = min(0.999, max(1e-6, (target * 5.0) / len(s)))
        lo = float(s.quantile(1.0 - frac))
    if hi is None:
        hi = float(s.max())

    best_thr, best_rate = hi, 0.0
    for _ in range(max_iter):
        mid = (lo + hi) / 2.0
        _, n_alerts = alert_times(preds, idcol, score_col, mid, silence_h)
        rate = 100.0 * n_alerts / bd
        best_thr, best_rate = mid, rate
        if abs(rate - target_per_100_beddays) <= tol * max(target_per_100_beddays, 1):
            break
        if n_alerts > target:
            lo = mid          # too many alerts, raise the threshold
        else:
            hi = mid
    return best_thr, best_rate

print("Alert simulation ready.")

Alert simulation ready.


In [6]:
# ---- calibrate on the observation clock, then hold the threshold fixed ----
# A threshold is chosen during development, on the data the developer has, and then
# deployed unchanged. Recalibrating it against the availability clock would describe a
# scenario nobody is in.
with Stage("threshold-calibration"):
    THR = {}
    rows = []
    for tag, C in COHORTS.items():
        P, idc = C["preds"], C["id"]
        for b in CFG.alert_budgets:
            thr, rate = calibrate_threshold(P, idc, SCORE_COLS["model_obs"], b, CFG.silence_hours)
            THR[(tag, b)] = thr
            rows.append({"cohort": tag, "budget_per_100_beddays": b,
                         "threshold": round(thr, 5), "achieved_rate": round(rate, 2),
                         "bed_days": round(bed_days(P), 1)})

cal = pd.DataFrame(rows)
display(cal)
write_table(cal, "t30_threshold_calibration")

# the bedside comparator needs its own threshold to hit the same workload
with Stage("threshold-calibration-comparator"):
    THR_NEWS = {}
    rows = []
    for tag, C in COHORTS.items():
        P, idc = C["preds"], C["id"]
        for b in CFG.alert_budgets:
            thr, rate = calibrate_threshold(P, idc, "news2", b, CFG.silence_hours)
            THR_NEWS[(tag, b)] = thr
            rows.append({"cohort": tag, "budget_per_100_beddays": b,
                         "news2_threshold": round(thr, 3), "achieved_rate": round(rate, 2)})

cal_n = pd.DataFrame(rows)
display(cal_n)
write_table(cal_n, "t31_threshold_calibration_news2")
print("\nNEWS2 is integer-valued, so its achievable alert rates are coarse. The nearest")
print("attainable rate is used and reported rather than forced to the exact budget.")

[threshold-calibration] start
[threshold-calibration] done in 2.4 s


,cohort,budget_per_100_beddays,threshold,achieved_rate,bed_days
0,MIMIC-IV test,5,0.532,5.040,"7,505.800"
1,MIMIC-IV test,10,0.313,10.070,"7,505.800"
2,MIMIC-IV test,20,0.185,19.680,"7,505.800"
3,MIMIC-IV test,40,0.106,40.120,"7,505.800"
4,MIMIC-IV test,80,0.063,79.020,"7,505.800"
5,eICU-CRD external,5,0.388,5.090,"78,846.200"
6,eICU-CRD external,10,0.303,9.990,"78,846.200"
7,eICU-CRD external,20,0.221,20.030,"78,846.200"
8,eICU-CRD external,40,0.153,39.890,"78,846.200"
9,eICU-CRD external,80,0.103,78.780,"78,846.200"


[threshold-calibration-comparator] start
[threshold-calibration-comparator] done in 13.4 s


,cohort,budget_per_100_beddays,news2_threshold,achieved_rate
0,MIMIC-IV test,5,11.000,2.080
1,MIMIC-IV test,10,10.000,5.890
2,MIMIC-IV test,20,9.000,13.870
3,MIMIC-IV test,40,8.000,29.860
4,MIMIC-IV test,80,7.000,102.550
5,eICU-CRD external,5,8.000,2.180
6,eICU-CRD external,10,7.000,7.350
7,eICU-CRD external,20,7.000,7.350
8,eICU-CRD external,40,6.000,20.570
9,eICU-CRD external,80,5.000,49.220



NEWS2 is integer-valued, so its achievable alert rates are coarse. The nearest
attainable rate is used and reported rather than forced to the exact budget.


---
## 3. Lead time and detection

For every stay with a documented escalation, the alert either fires before the event or it does not.
The scoring grid already stops at the event, so any alert a stay receives is by construction before
its escalation, and lead time is the interval between the two.

Four detection states follow from comparing the clocks:

* **Detected under both** — the alert fires under each clock; the question is how much later
* **Detected on the observation clock only** — the model would have caught this patient during
  development and does not catch them in deployment. This is the harm the study is about
* **Detected on the availability clock only** — the reverse, expected to be rare and reported for
  symmetry
* **Detected under neither**

Everything is computed at the primary alert budget, with the threshold fixed at its observation-clock
value.

In [7]:
def detection_table(preds, idcol, events, thr_obs, silence_h, max_lead, min_lead,
                    col_obs, col_av):
    """Per-stay first alert under each clock, joined to the escalation time."""
    fa_o, n_o = alert_times(preds, idcol, col_obs, thr_obs, silence_h)
    fa_a, n_a = alert_times(preds, idcol, col_av,  thr_obs, silence_h)

    base = (preds[[idcol, "first_event_h"]].drop_duplicates(idcol)
            .set_index(idcol))
    base["alert_obs_h"]   = fa_o
    base["alert_avail_h"] = fa_a
    base = base.reset_index()

    ev = base["first_event_h"]
    for clk in ("obs", "avail"):
        a = base[f"alert_{clk}_h"]
        lead = ev - a
        credited = a.notna() & ev.notna() & (lead > min_lead) & (lead <= max_lead)
        base[f"lead_{clk}_h"] = lead.where(credited)
        base[f"detected_{clk}"] = credited

    base["has_event"] = ev.notna()
    base["lead_lost_h"] = base["lead_obs_h"] - base["lead_avail_h"]
    return base, n_o, n_a


DET = {}
with Stage("detection-primary"):
    for tag, C in COHORTS.items():
        thr = THR[(tag, CFG.primary_budget)]
        d, n_o, n_a = detection_table(
            C["preds"], C["id"], C["events"], thr, CFG.silence_hours,
            CFG.max_lead_hours, CFG.min_lead_hours,
            SCORE_COLS["model_obs"], SCORE_COLS["model_avail"])
        DET[tag] = {"d": d, "n_alerts_obs": n_o, "n_alerts_avail": n_a,
                    "bed_days": bed_days(C["preds"]), "thr": thr}
        print(f"{tag}: {len(d):,} stays, {int(d['has_event'].sum()):,} with an escalation, "
              f"threshold {thr:.5f}")

[detection-primary] start
MIMIC-IV test: 6,098 stays, 847 with an escalation, threshold 0.18522
eICU-CRD external: 67,167 stays, 5,846 with an escalation, threshold 0.22058
[detection-primary] done in 0.2 s


In [8]:
def detection_summary(d, tag):
    e = d[d["has_event"]].copy()
    n = len(e)
    both  = e["detected_obs"] & e["detected_avail"]
    obs_o = e["detected_obs"] & ~e["detected_avail"]
    av_o  = ~e["detected_obs"] & e["detected_avail"]
    none  = ~e["detected_obs"] & ~e["detected_avail"]

    lost = e.loc[both, "lead_lost_h"]
    row = {
        "cohort": tag,
        "stays with escalation": f"{n:,}",
        "detected, observation clock": f"{int(e['detected_obs'].sum()):,} ({100*e['detected_obs'].mean():.1f}%)",
        "detected, availability clock": f"{int(e['detected_avail'].sum()):,} ({100*e['detected_avail'].mean():.1f}%)",
        "detected under both": f"{int(both.sum()):,} ({100*both.mean():.1f}%)",
        "lost to the availability clock": f"{int(obs_o.sum()):,} ({100*obs_o.mean():.2f}%)",
        "gained under availability clock": f"{int(av_o.sum()):,} ({100*av_o.mean():.2f}%)",
        "detected under neither": f"{int(none.sum()):,} ({100*none.mean():.1f}%)",
        "median lead, observation clock (h)":
            f"{e.loc[e['detected_obs'], 'lead_obs_h'].median():.1f}",
        "median lead, availability clock (h)":
            f"{e.loc[e['detected_avail'], 'lead_avail_h'].median():.1f}",
        "median lead lost among both-detected (h)": f"{lost.median():.2f}",
        "mean lead lost among both-detected (h)": f"{lost.mean():.2f}",
        "lost > 1 h": f"{int((lost > 1).sum()):,} ({100*(lost > 1).mean():.1f}%)",
        "lost > 2 h": f"{int((lost > 2).sum()):,} ({100*(lost > 2).mean():.1f}%)",
        "lost > 4 h": f"{int((lost > 4).sum()):,} ({100*(lost > 4).mean():.1f}%)",
    }
    return row

rows = [detection_summary(DET[t]["d"], t) for t in COHORTS]
det_tbl = pd.DataFrame(rows).set_index("cohort").T
display(det_tbl)
write_table(det_tbl, "t32_detection_and_lead_time", index=True)

cohort,MIMIC-IV test,eICU-CRD external
stays with escalation,847,"5,846"
"detected, observation clock",476 (56.2%),"1,711 (29.3%)"
"detected, availability clock",479 (56.6%),"1,643 (28.1%)"
detected under both,470 (55.5%),"1,594 (27.3%)"
lost to the availability clock,6 (0.71%),117 (2.00%)
gained under availability clock,9 (1.06%),49 (0.84%)
detected under neither,362 (42.7%),"4,086 (69.9%)"
"median lead, observation clock (h)",2.3,5.8
"median lead, availability clock (h)",2.3,5.8
median lead lost among both-detected (h),0.00,0.00


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t32_detection_and_lead_time.csv')

In [9]:
# ---- the number a unit would care about: per 100 stays --------------------
rows = []
for tag in COHORTS:
    D = DET[tag]; d = D["d"]
    e = d[d["has_event"]]
    n_stays = len(d)
    obs_only = int((e["detected_obs"] & ~e["detected_avail"]).sum())
    both = e["detected_obs"] & e["detected_avail"]
    lost = e.loc[both, "lead_lost_h"]

    rows.append({
        "cohort": tag,
        "stays": n_stays,
        "alerts, observation clock": D["n_alerts_obs"],
        "alerts, availability clock": D["n_alerts_avail"],
        "alerts per 100 bed-days, obs": round(100 * D["n_alerts_obs"] / D["bed_days"], 2),
        "alerts per 100 bed-days, avail": round(100 * D["n_alerts_avail"] / D["bed_days"], 2),
        "detections, obs": int(e["detected_obs"].sum()),
        "detections, avail": int(e["detected_avail"].sum()),
        "NNA, obs": round(D["n_alerts_obs"] / max(int(e["detected_obs"].sum()), 1), 1),
        "NNA, avail": round(D["n_alerts_avail"] / max(int(e["detected_avail"].sum()), 1), 1),
        "detections lost per 100 stays": round(100 * obs_only / n_stays, 3),
        "detections lost per 1000 stays": round(1000 * obs_only / n_stays, 2),
        "warning hours lost per 100 stays": round(100 * lost.sum() / n_stays, 1),
    })

burden = pd.DataFrame(rows).set_index("cohort").T
display(burden)
write_table(burden, "t33_alert_burden_and_loss", index=True)

print("\nNNA is the number of alerts issued per stay correctly warned. It is the quantity a unit")
print("trades against workload when choosing an operating point.")

cohort,MIMIC-IV test,eICU-CRD external
stays,"6,098.000","67,167.000"
"alerts, observation clock","1,477.000","15,796.000"
"alerts, availability clock","1,536.000","15,329.000"
"alerts per 100 bed-days, obs",19.680,20.030
"alerts per 100 bed-days, avail",20.460,19.440
"detections, obs",476.000,"1,711.000"
"detections, avail",479.000,"1,643.000"
"NNA, obs",3.100,9.200
"NNA, avail",3.200,9.300
detections lost per 100 stays,0.098,0.174



NNA is the number of alerts issued per stay correctly warned. It is the quantity a unit
trades against workload when choosing an operating point.


In [10]:
# ---- why detection can rise under the availability clock -------------------
# At a fixed threshold the two clocks do not issue the same number of alerts. Missing and
# stale laboratory features shift some scores, and where they shift upward the availability
# arm buys extra detection with extra workload. This quantifies that asymmetry so the
# direction of the detection difference can be read correctly.
rows = []
for tag, C in COHORTS.items():
    P = C["preds"]
    po = P[SCORE_COLS["model_obs"]].to_numpy()
    pa = P[SCORE_COLS["model_avail"]].to_numpy()
    ok = np.isfinite(po) & np.isfinite(pa)
    po, pa = po[ok], pa[ok]
    D = DET[tag]
    rows.append({
        "cohort": tag,
        "median score, obs": round(float(np.median(po)), 5),
        "median score, avail": round(float(np.median(pa)), 5),
        "mean score, obs": round(float(np.mean(po)), 5),
        "mean score, avail": round(float(np.mean(pa)), 5),
        "hours where avail score is higher (%)": round(100 * float((pa > po).mean()), 2),
        "hours where avail score is lower (%)": round(100 * float((pa < po).mean()), 2),
        "alerts, obs": D["n_alerts_obs"],
        "alerts, avail": D["n_alerts_avail"],
        "alert volume change (%)": round(100 * (D["n_alerts_avail"] / D["n_alerts_obs"] - 1), 2),
    })

asym = pd.DataFrame(rows).set_index("cohort").T
display(asym)
write_table(asym.reset_index().rename(columns={"index": "metric"}), "t45_alert_volume_asymmetry")

print("\nA fixed threshold does not hold workload fixed across the clocks. Where the availability")
print("arm issues more alerts it also detects more, so the detection difference at a fixed")
print("threshold confounds timing with volume. The budget sweep separates the two.")

cohort,MIMIC-IV test,eICU-CRD external
"median score, obs",0.014,0.034
"median score, avail",0.014,0.034
"mean score, obs",0.029,0.050
"mean score, avail",0.029,0.050
hours where avail score is higher (%),4.850,3.980
hours where avail score is lower (%),7.920,5.530
"alerts, obs","1,477.000","15,796.000"
"alerts, avail","1,536.000","15,329.000"
alert volume change (%),3.990,-2.960



A fixed threshold does not hold workload fixed across the clocks. Where the availability
arm issues more alerts it also detects more, so the detection difference at a fixed
threshold confounds timing with volume. The budget sweep separates the two.


---
## 4. Uncertainty

The paired lead time loss and the count of detections lost are the two numbers the paper turns on, so
both carry a stay-level bootstrap interval. Resampling at the level of the stay rather than the
scoring hour is what keeps the interval honest, since hours within a stay are not independent.

In [11]:
def bootstrap_ci(d, n_boot, seed):
    e = d[d["has_event"]].reset_index(drop=True)
    n = len(e)
    if n == 0:
        return {}
    rng = np.random.default_rng(seed)
    lead_lost, det_lost, med_obs, med_av = [], [], [], []
    both_mask = (e["detected_obs"] & e["detected_avail"]).to_numpy()
    lost_arr = e["lead_lost_h"].to_numpy()
    obs_only = (e["detected_obs"] & ~e["detected_avail"]).to_numpy()
    lo_arr = e["lead_obs_h"].to_numpy()
    la_arr = e["lead_avail_h"].to_numpy()

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        b = both_mask[idx]
        v = lost_arr[idx][b]
        lead_lost.append(np.nanmedian(v) if len(v) else np.nan)
        det_lost.append(100.0 * obs_only[idx].mean())
        med_obs.append(np.nanmedian(lo_arr[idx]))
        med_av.append(np.nanmedian(la_arr[idx]))

    def ci(a):
        a = np.asarray(a, dtype="float64")
        a = a[np.isfinite(a)]
        return (round(float(np.percentile(a, 2.5)), 3), round(float(np.percentile(a, 97.5)), 3))

    return {"median_lead_lost_h": ci(lead_lost),
            "detections_lost_pct_of_event_stays": ci(det_lost),
            "median_lead_obs_h": ci(med_obs),
            "median_lead_avail_h": ci(med_av)}


with Stage("bootstrap"):
    rows = []
    for tag in COHORTS:
        d = DET[tag]["d"]
        e = d[d["has_event"]]
        both = e["detected_obs"] & e["detected_avail"]
        ci = bootstrap_ci(d, CFG.n_boot, CFG.boot_seed)
        rows.append({
            "cohort": tag,
            "median lead, obs (h)": round(float(e.loc[e['detected_obs'], 'lead_obs_h'].median()), 2),
            "  95% CI": str(ci.get("median_lead_obs_h")),
            "median lead, avail (h)": round(float(e.loc[e['detected_avail'], 'lead_avail_h'].median()), 2),
            " 95% CI": str(ci.get("median_lead_avail_h")),
            "median lead lost (h)": round(float(e.loc[both, "lead_lost_h"].median()), 3),
            "95% CI": str(ci.get("median_lead_lost_h")),
            "detections lost, % of event stays":
                round(100 * float((e["detected_obs"] & ~e["detected_avail"]).mean()), 3),
            "95% CI ": str(ci.get("detections_lost_pct_of_event_stays")),
        })

boot = pd.DataFrame(rows).set_index("cohort").T
display(boot)
write_table(boot, "t34_bootstrap_intervals", index=True)
print(f"\n{CFG.n_boot:,} stay-level bootstrap replicates, percentile intervals.")

[bootstrap] start
[bootstrap] done in 0.8 s


cohort,MIMIC-IV test,eICU-CRD external
"median lead, obs (h)",2.270,5.850
95% CI,"(1.867, 2.617)","(5.433, 6.35)"
"median lead, avail (h)",2.270,5.850
95% CI,"(1.933, 2.65)","(5.383, 6.467)"
median lead lost (h),0.000,0.000
95% CI,"(0.0, 0.0)","(0.0, 0.0)"
"detections lost, % of event stays",0.708,2.001
95% CI,"(0.236, 1.299)","(1.659, 2.361)"



1,000 stay-level bootstrap replicates, percentile intervals.


---
## 5. The comparator

The bedside score is computed from vital signs, which reach the record when they are measured. It
therefore loses no lead time between the clocks, and that is the point of including it. Reporting it
at the same alert budget separates two questions that are usually conflated: how much warning the
learned model gives over the bedside score, and how much of that advantage survives the delay.

In [12]:
rows = []
with Stage("comparator-arm"):
    for tag, C in COHORTS.items():
        P, idc = C["preds"], C["id"]
        bd = bed_days(P)
        for name, col, thr in [
            ("Model, observation clock", SCORE_COLS["model_obs"],   THR[(tag, CFG.primary_budget)]),
            ("Model, availability clock", SCORE_COLS["model_avail"], THR[(tag, CFG.primary_budget)]),
            ("NEWS2 (latency-insensitive)", "news2", THR_NEWS[(tag, CFG.primary_budget)]),
        ]:
            fa, n_alerts = alert_times(P, idc, col, thr, CFG.silence_hours)
            b = (P[[idc, "first_event_h"]].drop_duplicates(idc).set_index(idc))
            b["a"] = fa
            b = b.reset_index()
            lead = b["first_event_h"] - b["a"]
            ok = b["a"].notna() & b["first_event_h"].notna() & \
                 (lead > CFG.min_lead_hours) & (lead <= CFG.max_lead_hours)
            ev = b["first_event_h"].notna()
            rows.append({
                "cohort": tag, "arm": name,
                "alerts per 100 bed-days": round(100 * n_alerts / bd, 2),
                "detection rate (%)": round(100 * ok.sum() / max(ev.sum(), 1), 2),
                "median lead (h)": round(float(lead[ok].median()), 2),
                "NNA": round(n_alerts / max(int(ok.sum()), 1), 1),
            })

arm = pd.DataFrame(rows)
display(arm)
write_table(arm, "t35_comparator_arm")

for tag in COHORTS:
    s = arm[arm["cohort"] == tag].set_index("arm")
    mo = s.loc["Model, observation clock"]
    ma = s.loc["Model, availability clock"]
    nw = s.loc["NEWS2 (latency-insensitive)"]
    print(f"\n{tag} at a matched alert budget:")
    print(f"  Detection rate  model obs {mo['detection rate (%)']:.1f}%  ->  "
          f"model avail {ma['detection rate (%)']:.1f}%  |  NEWS2 {nw['detection rate (%)']:.1f}%")
    print(f"  Median lead     model obs {mo['median lead (h)']:.1f} h  ->  "
          f"model avail {ma['median lead (h)']:.1f} h  |  NEWS2 {nw['median lead (h)']:.1f} h")
    adv_o = mo["detection rate (%)"] - nw["detection rate (%)"]
    adv_a = ma["detection rate (%)"] - nw["detection rate (%)"]
    print(f"  Detection advantage over the bedside score falls from "
          f"{adv_o:+.1f} to {adv_a:+.1f} percentage points.")

[comparator-arm] start
[comparator-arm] done in 0.4 s


,cohort,arm,alerts per 100 bed-days,detection rate (%),median lead (h),NNA
0,MIMIC-IV test,"Model, observation clock",19.680,56.200,2.270,3.100
1,MIMIC-IV test,"Model, availability clock",20.460,56.550,2.270,3.200
2,MIMIC-IV test,NEWS2 (latency-insensitive),13.870,11.690,6.780,10.500
3,eICU-CRD external,"Model, observation clock",20.030,29.270,5.850,9.200
4,eICU-CRD external,"Model, availability clock",19.440,28.100,5.850,9.300
5,eICU-CRD external,NEWS2 (latency-insensitive),7.350,8.450,7.110,11.700



MIMIC-IV test at a matched alert budget:
  Detection rate  model obs 56.2%  ->  model avail 56.5%  |  NEWS2 11.7%
  Median lead     model obs 2.3 h  ->  model avail 2.3 h  |  NEWS2 6.8 h
  Detection advantage over the bedside score falls from +44.5 to +44.9 percentage points.

eICU-CRD external at a matched alert budget:
  Detection rate  model obs 29.3%  ->  model avail 28.1%  |  NEWS2 8.4%
  Median lead     model obs 5.8 h  ->  model avail 5.8 h  |  NEWS2 7.1 h
  Detection advantage over the bedside score falls from +20.8 to +19.7 percentage points.


In [13]:
# ---- fair comparison: the model held to the bedside score's achievable workload ----
# NEWS2 is integer-valued, so it cannot be tuned to an arbitrary alert budget. Comparing
# the model at 20 alerts per 100 bed-days against NEWS2 at whatever rate it happens to
# reach gives the model more workload and therefore more detection. This scores the model
# at the rate NEWS2 actually achieves.
rows = []
with Stage("workload-matched-comparator"):
    for tag, C in COHORTS.items():
        P, idc = C["preds"], C["id"]
        rate_news = float(cal_n.loc[(cal_n["cohort"] == tag) &
                          (cal_n["budget_per_100_beddays"] == CFG.primary_budget),
                          "achieved_rate"].iloc[0])

        thr_m, got = calibrate_threshold(P, idc, SCORE_COLS["model_obs"],
                                         rate_news, CFG.silence_hours)
        d, n_o, n_a = detection_table(P, idc, C["events"], thr_m, CFG.silence_hours,
                                      CFG.max_lead_hours, CFG.min_lead_hours,
                                      SCORE_COLS["model_obs"], SCORE_COLS["model_avail"])
        e = d[d["has_event"]]

        a = arm[arm["cohort"] == tag].set_index("arm")
        n_row = a.loc["NEWS2 (latency-insensitive)"]

        rows.append({
            "cohort": tag,
            "matched alert rate (per 100 bed-days)": round(got, 2),
            "model detection, obs (%)": round(100 * e["detected_obs"].mean(), 2),
            "model detection, avail (%)": round(100 * e["detected_avail"].mean(), 2),
            "NEWS2 detection (%)": round(float(n_row["detection rate (%)"]), 2),
            "model median lead, obs (h)": round(float(e.loc[e["detected_obs"], "lead_obs_h"].median()), 2),
            "NEWS2 median lead (h)": round(float(n_row["median lead (h)"]), 2),
            "model NNA": round(n_o / max(int(e["detected_obs"].sum()), 1), 1),
            "NEWS2 NNA": round(float(n_row["NNA"]), 1),
            "advantage over NEWS2 (pp)": round(100 * e["detected_obs"].mean()
                                               - float(n_row["detection rate (%)"]), 2),
        })

matched = pd.DataFrame(rows).set_index("cohort").T
display(matched)
write_table(matched.reset_index().rename(columns={"index": "metric"}),
            "t44_model_at_news2_workload")

print("\nThis is the comparison to report against the bedside score. The unmatched figures in")
print("the previous cell give the model more alerts than NEWS2 can be tuned to issue.")

[workload-matched-comparator] start
[workload-matched-comparator] done in 0.4 s


cohort,MIMIC-IV test,eICU-CRD external
matched alert rate (per 100 bed-days),13.680,7.470
"model detection, obs (%)",50.770,15.240
"model detection, avail (%)",51.240,14.270
NEWS2 detection (%),11.690,8.450
"model median lead, obs (h)",1.840,4.930
NEWS2 median lead (h),6.780,7.110
model NNA,2.400,6.600
NEWS2 NNA,10.500,11.700
advantage over NEWS2 (pp),39.080,6.790



This is the comparison to report against the bedside score. The unmatched figures in
the previous cell give the model more alerts than NEWS2 can be tuned to issue.


---
## 6. Across the alert budget

A single operating point is a choice. The relationship between workload and detection is the thing a
unit actually needs, and it is where the two clocks can be compared without either being flattered by
a convenient threshold.

In [14]:
rows = []
with Stage("budget-sweep"):
    for tag, C in COHORTS.items():
        P, idc = C["preds"], C["id"]
        bd = bed_days(P)
        for b in CFG.alert_budgets:
            thr = THR[(tag, b)]
            d, n_o, n_a = detection_table(P, idc, C["events"], thr, CFG.silence_hours,
                                          CFG.max_lead_hours, CFG.min_lead_hours,
                                          SCORE_COLS["model_obs"], SCORE_COLS["model_avail"])
            e = d[d["has_event"]]
            both = e["detected_obs"] & e["detected_avail"]
            rows.append({
                "cohort": tag, "budget": b,
                "alerts per 100 bed-days, obs": round(100 * n_o / bd, 2),
                "alerts per 100 bed-days, avail": round(100 * n_a / bd, 2),
                "detection rate obs (%)": round(100 * e["detected_obs"].mean(), 2),
                "detection rate avail (%)": round(100 * e["detected_avail"].mean(), 2),
                "detection lost (pp)": round(100 * (e["detected_obs"].mean() -
                                                    e["detected_avail"].mean()), 3),
                "median lead obs (h)": round(float(e.loc[e["detected_obs"], "lead_obs_h"].median()), 2),
                "median lead avail (h)": round(float(e.loc[e["detected_avail"], "lead_avail_h"].median()), 2),
                "median lead lost (h)": round(float(e.loc[both, "lead_lost_h"].median()), 3),
                "NNA obs": round(n_o / max(int(e["detected_obs"].sum()), 1), 1),
            })

sweep = pd.DataFrame(rows)
display(sweep)
write_table(sweep, "t36_budget_sweep")

[budget-sweep] start
[budget-sweep] done in 1.3 s


,cohort,budget,"alerts per 100 bed-days, obs","alerts per 100 bed-days, avail",detection rate obs (%),detection rate avail (%),detection lost (pp),median lead obs (h),median lead avail (h),median lead lost (h),NNA obs
0,MIMIC-IV test,5,5.040,5.210,34.240,34.360,-0.118,1.410,1.430,0.000,1.300
1,MIMIC-IV test,10,10.070,10.430,45.340,45.570,-0.236,1.780,1.780,0.000,2.000
2,MIMIC-IV test,20,19.680,20.460,56.200,56.550,-0.354,2.270,2.270,0.000,3.100
3,MIMIC-IV test,40,40.120,40.640,66.120,65.760,0.354,2.880,2.850,0.000,5.400
4,MIMIC-IV test,80,79.020,78.530,73.790,73.440,0.354,3.400,3.380,0.000,9.500
5,eICU-CRD external,5,5.090,4.960,11.670,11.100,0.564,4.500,4.700,0.000,5.900
6,eICU-CRD external,10,9.990,9.680,18.700,17.890,0.804,5.070,5.280,0.000,7.200
7,eICU-CRD external,20,20.030,19.440,29.270,28.100,1.163,5.850,5.850,0.000,9.200
8,eICU-CRD external,40,39.890,38.950,41.340,39.990,1.351,6.520,6.490,0.000,13.000
9,eICU-CRD external,80,78.780,77.330,54.550,53.590,0.958,7.480,7.420,0.000,19.500


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t36_budget_sweep.csv')

---
## 7. Net benefit

Discrimination and workload each describe part of an operating point. Net benefit combines them at an
explicit exchange rate between a missed escalation and an unnecessary alert, which is the trade a
unit is making whether or not it says so. Reported across the range of thresholds a deployment might
plausibly adopt, under both clocks.

In [15]:
def net_benefit(y, p, pt):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(p)
    y, p, n = y[ok], p[ok], ok.sum()
    if n == 0:
        return np.nan
    flag = p >= pt
    tp = float(np.sum(flag & (y == 1)))
    fp = float(np.sum(flag & (y == 0)))
    return tp / n - (fp / n) * (pt / (1 - pt))


rows = []
with Stage("net-benefit"):
    for tag, C in COHORTS.items():
        P = C["preds"]
        y = P["y"].to_numpy()
        for pt in CFG.nb_thresholds:
            r = {"cohort": tag, "threshold_probability": pt,
                 "treat_all": float(np.mean(y) - (1 - np.mean(y)) * (pt / (1 - pt))),
                 "treat_none": 0.0}
            for nm, col in [("model_obs", SCORE_COLS["model_obs"]),
                            ("model_avail", SCORE_COLS["model_avail"])]:
                r[nm] = net_benefit(y, P[col].to_numpy(), pt)
            r["nb_lost"] = r["model_obs"] - r["model_avail"]
            rows.append(r)

nb = pd.DataFrame(rows)
nb[["treat_all", "model_obs", "model_avail", "nb_lost"]] = \
    nb[["treat_all", "model_obs", "model_avail", "nb_lost"]].round(6)
display(nb.head(20))
write_table(nb, "t37_net_benefit")

for tag in COHORTS:
    s = nb[nb["cohort"] == tag]
    i = s["nb_lost"].abs().idxmax()
    print(f"{tag}: largest net benefit difference {s.loc[i,'nb_lost']:+.5f} "
          f"at threshold probability {s.loc[i,'threshold_probability']:.3f}")

[net-benefit] start
[net-benefit] done in 2.2 s


,cohort,threshold_probability,treat_all,treat_none,model_obs,model_avail,nb_lost
0,MIMIC-IV test,0.005,0.022,0.000,0.022,0.022,0.000
1,MIMIC-IV test,0.010,0.017,0.000,0.019,0.019,-0.000
2,MIMIC-IV test,0.015,0.012,0.000,0.017,0.017,0.000
3,MIMIC-IV test,0.020,0.007,0.000,0.015,0.015,0.000
4,MIMIC-IV test,0.025,0.002,0.000,0.014,0.013,0.000
5,MIMIC-IV test,0.030,-0.003,0.000,0.013,0.012,0.000
6,MIMIC-IV test,0.035,-0.009,0.000,0.012,0.011,0.000
7,MIMIC-IV test,0.040,-0.014,0.000,0.011,0.011,0.000
8,MIMIC-IV test,0.045,-0.019,0.000,0.010,0.010,0.000
9,MIMIC-IV test,0.050,-0.024,0.000,0.010,0.010,0.000


MIMIC-IV test: largest net benefit difference +0.00019 at threshold probability 0.065
eICU-CRD external: largest net benefit difference +0.00012 at threshold probability 0.045


---
## 8. Culture results and the events they should have anticipated

Laboratory latency for chemistry is measured in tens of minutes. Microbiology is measured in days.
Part 1 found a median of 137 hours from collection to availability for blood cultures, and 76 hours
for those in which an organism was isolated.

That difference in magnitude changes what the question is. For chemistry the question is whether an
alert arrives an hour late. For cultures the question is whether the result arrives at all before the
clinical decision it was supposed to inform.

This section needs no model. It compares, for each stay, the hour a culture was collected, the hour
its result became available, and the hour the patient escalated.

In [16]:
with Stage("culture-gap"):
    ev_m = events_m.set_index("stay_id")["first_event_h"]
    mc = micro.copy()
    mc["latency_h"] = mc["latency_min"] / 60.0
    mc = mc[mc["latency_h"].notna() & (mc["latency_h"] >= 0) &
            (mc["latency_h"] <= 24 * 14) & (mc["hours_from_icu_admit"] >= 0)]
    mc["collect_h"] = mc["hours_from_icu_admit"]
    mc["result_h"] = mc["collect_h"] + mc["latency_h"]
    mc["first_event_h"] = mc["stay_id"].map(ev_m)

    # per stay: earliest culture collected before the escalation, and when it resulted
    pre = mc[mc["first_event_h"].notna() & (mc["collect_h"] < mc["first_event_h"])]
    g = (pre.sort_values("collect_h").groupby("stay_id")
         .agg(collect_h=("collect_h", "first"),
              result_h=("result_h", "first"),
              first_event_h=("first_event_h", "first"),
              organism=("organism_isolated", "max")))
    g["result_after_event"] = g["result_h"] > g["first_event_h"]
    g["gap_h"] = g["result_h"] - g["first_event_h"]

    pos = mc[mc["organism_isolated"] & mc["first_event_h"].notna() &
             (mc["collect_h"] < mc["first_event_h"])]
    gp = (pos.sort_values("collect_h").groupby("stay_id")
          .agg(collect_h=("collect_h", "first"), result_h=("result_h", "first"),
               first_event_h=("first_event_h", "first")))
    gp["result_after_event"] = gp["result_h"] > gp["first_event_h"]

cult = pd.DataFrame([
    {"population": "Any culture collected before escalation",
     "stays": f"{len(g):,}",
     "median collection to escalation (h)": f"{(g['first_event_h']-g['collect_h']).median():.1f}",
     "median collection to result (h)": f"{(g['result_h']-g['collect_h']).median():.1f}",
     "result arrived after the escalation": f"{int(g['result_after_event'].sum()):,} "
                                            f"({100*g['result_after_event'].mean():.1f}%)"},
    {"population": "Organism isolated, collected before escalation",
     "stays": f"{len(gp):,}",
     "median collection to escalation (h)": f"{(gp['first_event_h']-gp['collect_h']).median():.1f}",
     "median collection to result (h)": f"{(gp['result_h']-gp['collect_h']).median():.1f}",
     "result arrived after the escalation": f"{int(gp['result_after_event'].sum()):,} "
                                            f"({100*gp['result_after_event'].mean():.1f}%)"},
]).set_index("population").T
display(cult)
write_table(cult, "t38_culture_knowledge_gap", index=True)

print(f"\nAmong stays where a culture was collected before the escalation, the result was still")
print(f"unavailable when the escalation occurred in {100*g['result_after_event'].mean():.1f}% of cases.")
print("A model given culture status on the collection clock is credited with information that,")
print("in those stays, no clinician could have acted on.")

[culture-gap] start
[culture-gap] done in 0.1 s


population,Any culture collected before escalation,"Organism isolated, collected before escalation"
stays,"4,909","1,021"
median collection to escalation (h),5.1,6.9
median collection to result (h),57.8,64.9
result arrived after the escalation,"4,347 (88.6%)",874 (85.6%)



Among stays where a culture was collected before the escalation, the result was still
unavailable when the escalation occurred in 88.6% of cases.
A model given culture status on the collection clock is credited with information that,
in those stays, no clinician could have acted on.


In [17]:
# ---- how much of the record a culture-aware model would be given wrongly --
with Stage("culture-visibility"):
    rows = []
    for T in range(1, 49):
        collected = mc["collect_h"] <= T
        available = collected & (mc["result_h"] <= T)
        if collected.sum() < 50:
            continue
        rows.append({"scoring_hour": T, "cultures_collected": int(collected.sum()),
                     "results_available": int(available.sum()),
                     "available_pct": round(100 * available.sum() / collected.sum(), 2)})
    cvis = pd.DataFrame(rows)

display(cvis[cvis["scoring_hour"].isin([6, 12, 24, 48])])
write_table(cvis, "t39_culture_visibility_curve")
for T in (6, 12, 24, 48):
    r = cvis[cvis["scoring_hour"] == T]
    if len(r):
        print(f"  At hour {T:>2}: {r['available_pct'].iloc[0]:5.1f}% of collected cultures have resulted")

[culture-visibility] start
[culture-visibility] done in 0.1 s


,scoring_hour,cultures_collected,results_available,available_pct
5,6,63680,1260,1.980
11,12,85658,4853,5.670
23,24,114334,12172,10.650
47,48,146042,34610,23.700


  At hour  6:   2.0% of collected cultures have resulted
  At hour 12:   5.7% of collected cultures have resulted
  At hour 24:  10.7% of collected cultures have resulted
  At hour 48:  23.7% of collected cultures have resulted


---
## 9. Sensitivity

Three checks. The suppression window is an arbitrary operational choice and the results should not
turn on it. Turnaround class should order the effect if the mechanism is what we claim. And the
scoring interval is the reason the discrimination effect was small, so it is worth stating
explicitly rather than leaving the reader to infer it.

In [18]:
rows = []
with Stage("sensitivity-suppression"):
    for tag, C in COHORTS.items():
        P, idc = C["preds"], C["id"]
        for sil in CFG.silence_grid:
            thr, _ = calibrate_threshold(P, idc, SCORE_COLS["model_obs"],
                                         CFG.primary_budget, sil)
            d, n_o, n_a = detection_table(P, idc, C["events"], thr, sil,
                                          CFG.max_lead_hours, CFG.min_lead_hours,
                                          SCORE_COLS["model_obs"], SCORE_COLS["model_avail"])
            e = d[d["has_event"]]
            both = e["detected_obs"] & e["detected_avail"]
            rows.append({"cohort": tag, "suppression_h": sil,
                         "detection obs (%)": round(100 * e["detected_obs"].mean(), 2),
                         "detection avail (%)": round(100 * e["detected_avail"].mean(), 2),
                         "median lead lost (h)": round(float(e.loc[both, "lead_lost_h"].median()), 3),
                         "detections lost (%)": round(
                             100 * float((e["detected_obs"] & ~e["detected_avail"]).mean()), 3)})

sens = pd.DataFrame(rows)
display(sens)
write_table(sens, "t40_sensitivity_suppression")

[sensitivity-suppression] start
[sensitivity-suppression] done in 2.0 s


,cohort,suppression_h,detection obs (%),detection avail (%),median lead lost (h),detections lost (%)
0,MIMIC-IV test,2.000,51.480,52.540,0.000,0.472
1,MIMIC-IV test,4.000,56.200,56.550,0.000,0.708
2,MIMIC-IV test,8.000,59.030,59.390,0.000,0.708
3,eICU-CRD external,2.000,23.180,21.960,0.000,1.967
4,eICU-CRD external,4.000,29.270,28.100,0.000,2.001
5,eICU-CRD external,8.000,35.120,33.950,0.000,1.950


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t40_sensitivity_suppression.csv')

In [19]:
# ---- why the discrimination effect was small: scoring granularity ---------
labs_m = pd.read_parquet(DIR["cache"] / "labs_mimic.parquet",
                         columns=["turnaround_class", "latency_min"])
labs_m = labs_m[labs_m["latency_min"].between(0, 1440)]

rows = []
for cls, s in labs_m.groupby("turnaround_class")["latency_min"]:
    rows.append({"turnaround_class": cls, "n": len(s),
                 "median (min)": round(float(s.median()), 1),
                 "p90 (min)": round(float(s.quantile(0.90)), 1),
                 "p99 (min)": round(float(s.quantile(0.99)), 1),
                 "% resolved within 1 scoring interval (60 min)":
                     round(100 * float((s <= 60).mean()), 1),
                 "% delayed by 2+ intervals (>120 min)":
                     round(100 * float((s > 120).mean()), 1)})
gran = pd.DataFrame(rows).sort_values("median (min)").reset_index(drop=True)
display(gran)
write_table(gran, "t41_latency_versus_scoring_interval")

overall = labs_m["latency_min"]
print(f"\nAcross all analytes, {100*(overall <= 60).mean():.1f}% of results post within one hourly")
print(f"scoring interval and are therefore invisible to an hourly model for at most one cycle.")
print(f"{100*(overall > 120).mean():.1f}% are delayed by two or more intervals; those are the results")
print("that move alert timing, and microbiology sits entirely outside this range.")
del labs_m; gc.collect()

,turnaround_class,n,median (min),p90 (min),p99 (min),% resolved within 1 scoring interval (60 min),% delayed by 2+ intervals (>120 min)
0,poc,1247860,3.000,6.000,19.000,99.800,0.100
1,core,5241236,59.000,102.000,224.000,53.200,5.600
2,send,89251,72.000,217.000,720.000,37.100,22.400



Across all analytes, 61.8% of results post within one hourly
scoring interval and are therefore invisible to an hourly model for at most one cycle.
4.8% are delayed by two or more intervals; those are the results
that move alert timing, and microbiology sits entirely outside this range.


0

---
## 10. Figures

In [20]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
})
C_OBS, C_AV, C_REF = "#2b6cb0", "#c05621", "#4a5568"

# --- F11: paired lead time distribution -------------------------------------
fig, axes = plt.subplots(1, len(COHORTS), figsize=(11, 4.0), squeeze=False)
for ax, tag in zip(axes[0], COHORTS):
    e = DET[tag]["d"]
    lo = e.loc[e["detected_obs"], "lead_obs_h"].dropna()
    la = e.loc[e["detected_avail"], "lead_avail_h"].dropna()
    bins = np.arange(0, CFG.max_lead_hours + 2, 2)
    ax.hist(lo, bins=bins, alpha=0.55, color=C_OBS, label=f"Observation clock (n={len(lo):,})")
    ax.hist(la, bins=bins, alpha=0.55, color=C_AV,  label=f"Availability clock (n={len(la):,})")
    ax.axvline(lo.median(), color=C_OBS, ls="--", lw=1.4)
    ax.axvline(la.median(), color=C_AV,  ls="--", lw=1.4)
    ax.set_xlabel("Lead time before escalation (hours)")
    ax.set_ylabel("Stays")
    ax.set_title(tag)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_fig(fig, "f11_lead_time_distribution")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f11_lead_time_distribution.png')

In [21]:
# --- F12: per-patient lead time lost ----------------------------------------
fig, axes = plt.subplots(1, len(COHORTS), figsize=(11, 3.8), squeeze=False)
for ax, tag in zip(axes[0], COHORTS):
    e = DET[tag]["d"]
    both = e["detected_obs"] & e["detected_avail"]
    lost = e.loc[both, "lead_lost_h"].dropna()
    lost = lost[np.isfinite(lost)]
    ax.hist(lost, bins=np.arange(-2, 13, 0.5), color=C_OBS, alpha=0.8)
    ax.axvline(0, color=C_REF, lw=1.0, ls=":")
    ax.axvline(lost.median(), color=C_AV, lw=1.6, ls="--",
               label=f"median {lost.median():.2f} h")
    ax.set_xlabel("Warning hours lost per patient")
    ax.set_ylabel("Stays")
    ax.set_title(f"{tag}\n{100*(lost > 1).mean():.1f}% lose more than one hour")
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_fig(fig, "f12_lead_time_lost")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f12_lead_time_lost.png')

In [22]:
# --- F13: workload against detection ----------------------------------------
fig, axes = plt.subplots(1, len(COHORTS), figsize=(11, 4.0), squeeze=False)
for ax, tag in zip(axes[0], COHORTS):
    s = sweep[sweep["cohort"] == tag]
    ax.plot(s["alerts per 100 bed-days, obs"], s["detection rate obs (%)"],
            "o-", color=C_OBS, lw=2, label="Observation clock")
    ax.plot(s["alerts per 100 bed-days, avail"], s["detection rate avail (%)"],
            "s--", color=C_AV, lw=2, label="Availability clock")
    n = arm[(arm["cohort"] == tag) & (arm["arm"].str.startswith("NEWS2"))]
    if len(n):
        ax.plot(n["alerts per 100 bed-days"], n["detection rate (%)"], "D",
                color=C_REF, ms=8, label="NEWS2 (latency-insensitive)")
    ax.set_xlabel("Alerts per 100 occupied bed-days")
    ax.set_ylabel("Escalations detected (%)")
    ax.set_title(tag)
    ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
save_fig(fig, "f13_workload_detection")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f13_workload_detection.png')

In [23]:
# --- F14: net benefit --------------------------------------------------------
fig, axes = plt.subplots(1, len(COHORTS), figsize=(11, 4.0), squeeze=False)
for ax, tag in zip(axes[0], COHORTS):
    s = nb[nb["cohort"] == tag]
    ax.plot(s["threshold_probability"], s["model_obs"], color=C_OBS, lw=2, label="Observation clock")
    ax.plot(s["threshold_probability"], s["model_avail"], color=C_AV, lw=2, ls="--",
            label="Availability clock")
    ax.plot(s["threshold_probability"], s["treat_all"], color=C_REF, lw=1.2, ls=":", label="Alert on all")
    ax.axhline(0, color=C_REF, lw=1.0)
    ax.set_ylim(bottom=min(-0.005, s[["model_obs", "model_avail"]].min().min()))
    ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.set_title(tag)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_fig(fig, "f14_net_benefit")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f14_net_benefit.png')

In [24]:
# --- F15: culture knowledge gap ---------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

ax = axes[0]
ax.plot(cvis["scoring_hour"], cvis["available_pct"], lw=2.4, color="#553c9a")
ax.axhline(100, color=C_REF, lw=0.8, ls=":")
ax.set_xlabel("Scoring hour after ICU admission")
ax.set_ylabel("Collected cultures that have resulted (%)")
ax.set_title("Culture result availability")
ax.set_ylim(0, 100)

ax = axes[1]
gap = (g["result_h"] - g["first_event_h"]).dropna()
gap = gap[np.isfinite(gap)]
ax.hist(np.clip(gap, -72, 168), bins=40, color="#553c9a", alpha=0.85)
ax.axvline(0, color="#c53030", lw=1.8,
           label=f"escalation  |  {100*(gap > 0).mean():.1f}% resulted afterwards")
ax.set_xlabel("Hours from escalation to culture result (positive = after)")
ax.set_ylabel("Stays")
ax.set_title("When the culture result arrived, relative to the escalation")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_fig(fig, "f15_culture_knowledge_gap")

WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f15_culture_knowledge_gap.png')

---
## 11. Manuscript summary

In [28]:
summary_rows = []
for tag in COHORTS:
    D = DET[tag]; d = D["d"]; e = d[d["has_event"]]
    both = e["detected_obs"] & e["detected_avail"]
    lost = e.loc[both, "lead_lost_h"]
    a = arm[arm["cohort"] == tag].set_index("arm")
    summary_rows.append({
        "cohort": tag,
        "Stays": f"{len(d):,}",
        "Stays with escalation": f"{len(e):,}",
        "Alert budget (per 100 bed-days)": f"{CFG.primary_budget}",
        "Detection rate, observation clock": f"{100*e['detected_obs'].mean():.1f}%",
        "Detection rate, availability clock": f"{100*e['detected_avail'].mean():.1f}%",
        "Detection rate, NEWS2 (at its own workload)":
            f"{a.loc['NEWS2 (latency-insensitive)','detection rate (%)']:.1f}%",
        "Model detection at NEWS2 workload":
            f"{matched.loc['model detection, obs (%)', tag]:.1f}%",
        "Median lead, observation clock":
            f"{e.loc[e['detected_obs'],'lead_obs_h'].median():.1f} h",
        "Median lead, availability clock":
            f"{e.loc[e['detected_avail'],'lead_avail_h'].median():.1f} h",
        "Median warning lost per patient": f"{lost.median():.2f} h",
        "Patients losing > 1 h of warning": f"{100*(lost > 1).mean():.1f}%",
        "Patients losing > 4 h of warning": f"{100*(lost > 4).mean():.1f}%",
        "Detections lost per 1000 stays":
            f"{1000 * int((e['detected_obs'] & ~e['detected_avail']).sum()) / len(d):.2f}",
    })

final = pd.DataFrame(summary_rows).set_index("cohort").T
display(final)
write_table(final, "t42_manuscript_summary", index=True)

cohort,MIMIC-IV test,eICU-CRD external
Stays,"6,098","67,167"
Stays with escalation,847,"5,846"
Alert budget (per 100 bed-days),20,20
"Detection rate, observation clock",56.2%,29.3%
"Detection rate, availability clock",56.6%,28.1%
"Detection rate, NEWS2 (at its own workload)",11.7%,8.4%
Model detection at NEWS2 workload,50.8%,15.2%
"Median lead, observation clock",2.3 h,5.8 h
"Median lead, availability clock",2.3 h,5.8 h
Median warning lost per patient,0.00 h,0.00 h


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t42_manuscript_summary.csv')

In [26]:
checks = []
tag0 = "MIMIC-IV test"
e0 = DET[tag0]["d"]; e0 = e0[e0["has_event"]]
both0 = e0["detected_obs"] & e0["detected_avail"]
lost0 = e0.loc[both0, "lead_lost_h"]

checks.append(("Alert budgets achieved within tolerance",
               bool((cal["achieved_rate"] - cal["budget_per_100_beddays"]).abs().max() <
                    0.10 * cal["budget_per_100_beddays"].max()),
               f"max deviation {float((cal['achieved_rate']-cal['budget_per_100_beddays']).abs().max()):.2f}"))

checks.append(("Lead time is lost in the expected direction",
               bool(lost0.median() >= 0),
               f"median {lost0.median():.2f} h"))

checks.append(("A meaningful minority lose more than an hour",
               bool((lost0 > 1).mean() > 0.02),
               f"{100*(lost0 > 1).mean():.1f}% of both-detected stays"))

checks.append(("The model beats the bedside score on detection",
               bool(arm.loc[(arm['cohort'] == tag0) &
                            (arm['arm'] == 'Model, observation clock'), 'detection rate (%)'].iloc[0] >
                    arm.loc[(arm['cohort'] == tag0) &
                            (arm['arm'].str.startswith('NEWS2')), 'detection rate (%)'].iloc[0]),
               "model versus NEWS2 at matched workload"))

checks.append(("Culture results frequently arrive after the escalation",
               bool(g["result_after_event"].mean() > 0.20),
               f"{100*g['result_after_event'].mean():.1f}% of stays"))

checks.append(("Findings are stable across suppression windows",
               bool(sens[sens['cohort'] == tag0]["median lead lost (h)"].std() < 1.0),
               f"sd {float(sens[sens['cohort']==tag0]['median lead lost (h)'].std()):.3f} h"))

res = pd.DataFrame(checks, columns=["check", "passed", "observed"])
res["status"] = np.where(res["passed"], "PASS", "REVIEW")
display(res[["check", "status", "observed"]])
write_table(res, "t43_part3_gate_checks")

,check,status,observed
0,Alert budgets achieved within tolerance,PASS,max deviation 1.22
1,Lead time is lost in the expected direction,PASS,median 0.00 h
2,A meaningful minority lose more than an hour,REVIEW,1.1% of both-detected stays
3,The model beats the bedside score on detection,PASS,model versus NEWS2 at matched workload
4,Culture results frequently arrive after the es...,PASS,88.6% of stays
5,Findings are stable across suppression windows,PASS,sd 0.000 h


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t43_part3_gate_checks.csv')

In [27]:
manifest = PROV.manifest()
manifest["prior_chains"] = PRIOR
manifest["part3_summary"] = {
    "primary_alert_budget_per_100_beddays": CFG.primary_budget,
    "suppression_hours": CFG.silence_hours,
    "cohorts": {},
    "culture_result_after_event_pct": float(100 * g["result_after_event"].mean()),
    "culture_positive_result_after_event_pct": float(100 * gp["result_after_event"].mean()),
}
for tag in COHORTS:
    d = DET[tag]["d"]; e = d[d["has_event"]]
    both = e["detected_obs"] & e["detected_avail"]
    lost = e.loc[both, "lead_lost_h"]
    manifest["part3_summary"]["cohorts"][tag] = {
        "stays": int(len(d)),
        "event_stays": int(len(e)),
        "detection_rate_obs": float(e["detected_obs"].mean()),
        "detection_rate_avail": float(e["detected_avail"].mean()),
        "median_lead_obs_h": float(e.loc[e["detected_obs"], "lead_obs_h"].median()),
        "median_lead_avail_h": float(e.loc[e["detected_avail"], "lead_avail_h"].median()),
        "median_lead_lost_h": float(lost.median()),
        "pct_losing_over_1h": float(100 * (lost > 1).mean()),
        "pct_losing_over_4h": float(100 * (lost > 4).mean()),
        "detections_lost_pct": float(100 * (e["detected_obs"] & ~e["detected_avail"]).mean()),
        "alerts_per_100_beddays_obs": float(100 * DET[tag]["n_alerts_obs"] / DET[tag]["bed_days"]),
    }

MANIFEST_P = OUT / "part3_manifest.json"
with open(MANIFEST_P, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Manifest written: {MANIFEST_P}")
print(f"Terminal chain digest: {PROV.chain}")
print(f"\nTables: {sum(1 for r in PROV.records if r['role']=='table')} | "
      f"Figures: {sum(1 for r in PROV.records if r['role']=='figure')}")

t = pd.DataFrame([{"stage": k, "seconds": v} for k, v in PROV.timings.items()])
if len(t):
    display(t.sort_values("seconds", ascending=False).reset_index(drop=True))
print("\nPart 3 complete. The three manifests form one hash-linked chain from the source CSVs")
print("to every table and figure in the analysis.")

Manifest written: C:\Research_Paper_2\result_availability_audit\part3_manifest.json
Terminal chain digest: f9b480507372fe29c2b4f122ed8616991c1ef24ea476b6430daa4b8eb9d1076d

Tables: 16 | Figures: 5


,stage,seconds
0,threshold-calibration-comparator,13.350
1,threshold-calibration,2.420
2,net-benefit,2.160
3,sensitivity-suppression,1.980
4,budget-sweep,1.340
5,bootstrap,0.830
6,load,0.620
7,comparator-arm,0.390
8,workload-matched-comparator,0.350
9,detection-primary,0.250



Part 3 complete. The three manifests form one hash-linked chain from the source CSVs
to every table and figure in the analysis.
